# Pedal Poll / Sondo Velo data cleaning

This notebook loads data and performs cleaning steps to produce a final dataset for analysis.

- Pedal poll 2021: June 1-6 2021
- Pedal poll 2022: June 7-12 2022
- Pedal poll 2023: June 6-11 2023 with option to count following week for smoke
- Pedal poll 2024: June 4-9 2024

At each official count location, the goal was for 120-minute long counts during four weekday periods (07–09:00, 11–13:00, 14–16:00 and 16–18:00) and one weekend period (12–14:00).

## Cleaning steps:

- [x] remove negative counts (2023 and 2024 data)
- [x] remove sessions outside the count window unless they are between 1 and 3 hours and take place during one of the designated count time windows or have the "Pedal Poll Canada" team name
- [x] remove sessions lasting less than 5 minutes (as in 2021)
- [x] remove sessions lasting more than 4 hours (as in 2021)
- [x] remove duplicate count sessions - doesn't seem to have happened in 2023 or 2024 data
- [x] add paper count data
- [x] remove duplicated app and hand count data 

2022 only (based on 2021 process):
- [x] replace counts with values other than 1 or NULL in demographic columns within unknown - gender, age and ethnicity entries with values from 2 to 5
- [x] create demographic summary column

Counterpoint IDs were grouped to merge counterpoints within 100 m of each other (counterpoint_location_grouping.md).

## Data notes

In each year the process of acquiring the raw data is slightly different as the app backend continues to evolve. In each year tabulated paper counts were provided by VCB.

In 2022 demographic data was collected, so this script is for 2023 onwards ONLY.

### 2022

Data were downloaded from the Counterpoint Data Explorer for June 2022.

### 2023 and 2024:

Data for June 2023 were downloaded using the following query: https://superset.counterpointapp.org/superset/explore/p/Nk5eQpXYOZR/
Data for June 2024 were downloaded using the following query: https://superset.counterpointapp.org/superset/dashboard/p/Bb80wKA0kl3/

Steps to download from Counterpoint:
 - Three vertical dots -> "edit chart"
- Bottom of second column from left:
- -> turn off server pagination
- -> row limit 50000
- Horizontal dots on the right
- -> Download -> Export to CSV

In [1]:
# set year 
year = 2024

In [2]:
# load packages
import pandas as pd
import numpy as np

In [3]:
if year == 2023:
    filename = "data/20250624_142657.csv" 
if year == 2024:
    filename = "data/20240916_153200.csv"
    
app_data = pd.read_csv(filename)

In [4]:
rows_removed = 0
original_shape = app_data.shape[0]

In [5]:
# keep only counts from Canada
print(app_data['counterpoint_geocoded_country'].unique())
app_data_filtered = app_data[app_data['counterpoint_geocoded_country'] == 'Canada']

['Canada']


In [6]:
print("%s count rows were removed from outside Canada"  %(app_data.shape[0] - app_data_filtered.shape[0]))
rows_removed += app_data.shape[0] - app_data_filtered.shape[0]

0 count rows were removed from outside Canada


In [7]:
# which dates have counts?
app_data_filtered['count_session_start_datetime_dayofmonth'].value_counts()

count_session_start_datetime_dayofmonth
4     22149
8      7064
5      2538
6      1810
9       822
3       231
7       207
30       15
2         2
Name: count, dtype: int64

In [8]:
# if a column has all zeros or all na values, drop it
# get the columns that are not zero and not na
data_columns = app_data_filtered.columns[(~np.all(app_data_filtered == 0, axis = 0))
                         & (~np.all(app_data_filtered.isna(), axis = 0))]

print("%s columns were dropped with %s remaining" %(app_data_filtered.shape[1] - len(data_columns), 
                                                    len(data_columns)))
                                                    
# keep only the non-zero and non-na columns
app_data_filtered = app_data_filtered.loc[:,data_columns]

18 columns were dropped with 74 remaining


In [9]:
# these columns are the same for each unique count session
count_session_columns = ['counterpoint_id', 'counterpoint_name', 'counterpoint_lat',
       'counterpoint_lng', 'counterpoint_coordinates', 'counterpoint_geocoded_city',
       'counterpoint_geocoded_state', 'counterpoint_geocoded_country',
       'count_session_id',
       'count_session_start_datetime', 'count_session_end_datetime',
       'count_session_start_datetime_year',
       'count_session_start_datetime_month',
       'count_session_start_datetime_month_number',
       'count_session_start_datetime_dayofmonth',
       'count_session_start_datetime_dayofweek',
       'count_session_start_datetime_dayofweek_number',
       'count_session_start_datetime_hourofday',
       'count_session_elapsed_time_seconds',
       'count_session_elapsed_time_minutes', 'team_id', 'team_name']

In [10]:
# unique count sessions
print("Before cleaning, there are %s unique count sessions" 
      %app_data_filtered[count_session_columns].drop_duplicates().shape[0])

Before cleaning, there are 410 unique count sessions


Due to an app error, some negative per-minute counts were recorded in the 2023 and 2024 app data; the negative counts were set to zero

In [11]:
num_negative = np.sum(np.any(app_data_filtered[['easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']] < 0, axis = 1))

print("There are N=%s rows with negative counts" %num_negative)

There are N=65 rows with negative counts


In [12]:
# save negative counts to send to developers
app_data_filtered[np.any(app_data_filtered[['easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']] < 0, axis = 1)].to_csv("negative_counts_%s.csv" %year, index = None)

In [13]:
# set negative count values to zero
app_data_filtered[['easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']] = app_data_filtered[['easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']].clip(lower=0)

In [14]:
print("The total count of bikes before further cleaning is %s" %np.sum(app_data_filtered['easy_bike_total']))

The total count of bikes before further cleaning is 45282


We removed all count sessions less than 5 minutes to exclude practice counts and counts longer than 4 hours. In 2023 and 2024 no counts were longer than 4 hours.

In [15]:
original_count_minutes = app_data_filtered.shape[0]

In [16]:
short_sessions = app_data_filtered[(app_data_filtered['count_session_elapsed_time_seconds'] < 300)]

In [17]:
long_sessions = app_data_filtered[(app_data_filtered['count_session_elapsed_time_seconds'] > 14400)]

In [18]:
app_data_filtered = app_data_filtered[(app_data_filtered['count_session_elapsed_time_seconds'] >= 300)
                 & (app_data_filtered['count_session_elapsed_time_seconds'] <= 14400)]

In [19]:
rows_removed += short_sessions.shape[0]
rows_removed += long_sessions.shape[0]
print("%s rows were removed from short counts and %s rows from long counts." %(len(short_sessions), 
                                                                                  len(long_sessions)))

631 rows were removed from short counts and 0 rows from long counts.


Keep data from outside the official count week only if it has the Pedal Poll Canada team name OR it is around 2 hours long AND takes place during one of the designated count times

In [20]:
if year == 2023:
    start_date = 6
    end_date = 11
if year == 2024:
    start_date = 4
    end_date = 9

In [21]:
after_week_data = app_data_filtered[app_data_filtered['count_session_start_datetime_dayofmonth'] > end_date]
before_week_data = app_data_filtered[app_data_filtered['count_session_start_datetime_dayofmonth'] < start_date]

In [22]:
# this captures if someone starts their 7 am count at 6:59, for instance
valid_weekday_start_hours = [6,7, 10,11, 13,14, 15,16]
valid_weekend_start_hours = [11,12]
threshold_from_2_hours = 60 # minutes - keep counts between 1 and 3 hours

In [23]:
# data from outside the designated counting week
outside_data = app_data_filtered[(app_data_filtered['count_session_start_datetime_dayofmonth'] > end_date)
                 | (app_data_filtered['count_session_start_datetime_dayofmonth'] < start_date)]

In [24]:
valid_counts = outside_data[
            # weekend data
            (((outside_data['count_session_start_datetime_dayofweek'] == 'Saturday')
              | (outside_data['count_session_start_datetime_dayofweek'] == 'Sunday'))
             & (outside_data['count_session_start_datetime_hourofday'].isin(valid_weekend_start_hours)))
            # weekday data
           | (((outside_data['count_session_start_datetime_dayofweek'] != 'Saturday')
              & (outside_data['count_session_start_datetime_dayofweek'] != 'Sunday'))
             & (outside_data['count_session_start_datetime_hourofday'].isin(valid_weekday_start_hours)))
            # pedal poll team name
           | (outside_data['team_name'] == 'Pedal Poll Canada')
]

In [25]:
# remove if not between 1 and 3 hours long
valid_counts = valid_counts[np.abs(valid_counts['count_session_elapsed_time_minutes'] - 120) <= threshold_from_2_hours]

In [26]:
# check - do retained counts make sense?
valid_counts.drop_duplicates(subset = 'count_session_id')[['count_session_elapsed_time_minutes', 
                                                           'count_session_start_datetime_dayofmonth', 
                                                     'count_session_start_datetime_hourofday', 
                                                           'count_session_start_datetime_dayofweek', 
                                                     'count_session_start_datetime_dayofweek_number', 'team_name',
                                                          'counterpoint_name']]

,count_session_elapsed_time_minutes,count_session_start_datetime_dayofmonth,count_session_start_datetime_hourofday,count_session_start_datetime_dayofweek,count_session_start_datetime_dayofweek_number,team_name,counterpoint_name
34723,119,3,6,Monday,1,Pedal Poll Canada,Inverness


In [27]:
primary_data = app_data_filtered[(app_data_filtered['count_session_start_datetime_dayofmonth'] >= start_date)
                 & (app_data_filtered['count_session_start_datetime_dayofmonth'] <= end_date)]

In [28]:
primary_data.shape

(33962, 74)

In [29]:
rows_removed += app_data_filtered.shape[0] - primary_data.shape[0]
rows_removed -= valid_counts.shape[0]

In [30]:
app_data_filtered = pd.concat([primary_data, valid_counts])

In [31]:
app_data_filtered.shape

(34075, 74)

In [32]:
print("%s rows (count minutes) were added from outside the official count window" 
      %(app_data_filtered.shape[0] - primary_data.shape[0]))

113 rows (count minutes) were added from outside the official count window


Data from June but outside the official poll count week were retained if they had the official team name "Pedal Poll Canada" or if they began during one of the specified weekday or weekend count intervals and were between 1 and 3 hours long. This was to account for some counts being delayed due to weather (in particular wildfire smoke in 2023). 

This added 1304 count minutes to the dataset for 2023. In 2024 this added a single count of 113 minutes from Monday, June 3.

In [33]:
# limit columns some more to match 2021 data
app_data_filtered = app_data_filtered[['counterpoint_id', 'counterpoint_name', 'counterpoint_lat',
       'counterpoint_lng', 'counterpoint_coordinates',
       'counterpoint_geocoded_city',
       'counterpoint_geocoded_state',
       'count_session_id',
       'count_session_start_datetime', 'count_session_end_datetime',
       'count_session_start_datetime_year',
       'count_session_start_datetime_month',
       'count_session_start_datetime_month_number',
       'count_session_start_datetime_dayofmonth',
       'count_session_start_datetime_dayofweek',
       'count_session_start_datetime_dayofweek_number',
       'count_session_start_datetime_hourofday',
       'count_session_elapsed_time_seconds',
       'count_session_elapsed_time_minutes', 'team_id', 'team_name',
       'count_time', 'easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']]

In [34]:
# rename columns to match 2021 data
app_data_filtered.columns = ['counterpoint_id', 'counterpoint_name', 'lat',
       'lng', 'coordinates',
        'geocoded_city', 'geocoded_province', 'survey_session_id',
       'start_datetime', 'end_datetime',
       'start_datetime_year',
       'start_datetime_month',
       'start_datetime_month_number',
       'start_datetime_dayofmonth',
       'start_datetime_dayofweek',
       'start_datetime_dayofweek_number',
       'start_datetime_hourofday',
       'elapsed_time_seconds',
       'elapsed_time_minutes', 'team_id', 'team_name',
       'count_time', 'easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']

In [35]:
app_data_agg = app_data_filtered.groupby(['counterpoint_id', 'counterpoint_name', 'survey_session_id', 'lat', 
                         'lng', 'geocoded_city', 'start_datetime', 'start_datetime_hourofday', 
                                          'elapsed_time_seconds'])[['easy_bike_total', 'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']].sum().reset_index()

In [36]:
count_session_columns = ['counterpoint_id', 'counterpoint_name', 'lat', 'lng', 'coordinates',
       'geocoded_city', 'geocoded_province', 'survey_session_id',
       'start_datetime', 'end_datetime', 'start_datetime_year',
       'start_datetime_month', 'start_datetime_month_number',
       'start_datetime_dayofmonth', 'start_datetime_dayofweek',
       'start_datetime_dayofweek_number', 'start_datetime_hourofday',
       'elapsed_time_seconds', 'elapsed_time_minutes', 'team_id', 'team_name']

In [37]:
# unique count sessions after cleaning
print("There are %s unique count sessions after cleaning" 
      %app_data_filtered[count_session_columns].drop_duplicates().shape[0])

There are 384 unique count sessions after cleaning


In [38]:
assert app_data_filtered.shape[0] == original_shape - rows_removed

## Add paper count data

Note: in 2024 there was one paper count that was recorded as being counted by a pair of people, each counting in a different direction. Check in analysis processing that this is processed correctly.

I removed identifying information in the location notes column and condensed location and location notes into a single column.

In [39]:
if year == 2023:
    paper_count_fn = "data/2023_hand_count.csv"
    brampton_hand_count_2023 = pd.read_csv("data/brampton_2023_paper.csv")
elif year == 2024: 
    paper_count_fn = "data/2024_hand_count.csv"
paper_count = pd.read_csv(paper_count_fn)

In [40]:
rows_removed = 0
original_shape = paper_count.shape[0]

In [41]:
original_shape

32

In [42]:
# drop the first example row in 2023
if year == 2023:
    paper_count = paper_count.iloc[1:]
    rows_removed += 1

In [43]:
# drop columns with all NA
non_na_columns = paper_count.columns[~np.all(paper_count.isna(), axis = 0)]

paper_count = paper_count[non_na_columns]

In [44]:
# the counterpoint columns are replacements for the manual columns; in some cases that data was recorded as more accurate.
# merge with the paper count columns and check later for duplicates.
if year == 2024:
    column_pairs = {'Cyclists': 'CounterPoint: \nCyclists',
                    'Pedestrians': 'CounterPoint:\nPedestrians',
                    'Personal vehicles': 'CounterPoint:\nPersonal vehicles',
                    'Over sized vehicles': 'CounterPoint:\nOver sized vehicles',
                    'Manual Count: Start time \n(YYYY-MM-DD HH:MM:SS)': 'CounterPoint App data: \nStart time(s)\n(YYYY-MM-DD HH:MM:SS)',
                   'Manual Count: End time\n(YYYY-MM-DD HH:MM:SS)': 'CounterPoint App data: \nEnd time(s)\n(YYYY-MM-DD HH:MM:SS)'}

    for paper_col, app_col in column_pairs.items():
        # check that at least one column has an NA for each row
        assert np.all(np.sum(paper_count[[paper_col, app_col]].isna(), axis =1) >= 1)
        paper_count[paper_col] = paper_count[paper_col].fillna(paper_count[app_col])
        
    paper_count = paper_count.drop(columns = list(column_pairs.values()))

In [45]:
# CAUTION: check column format each year
if year == 2024:
    paper_count.columns = ['geocoded_city', 'geocoded_province', 'counterpoint_id',
       'counterpoint_name', 'date',
       'location_notes', 
       'start_datetime',
       'end_datetime', 'easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total', 
       'notes']

if year == 2023:
    paper_count.columns = ['geocoded_city', 'geocoded_province', 'counterpoint_id',
       'counterpoint_name', 'date',
       'location_notes', 'team_name',
       'start_datetime',
       'end_datetime', 'easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total', 
       'notes', 'counterpoint_start_datetime']

In [46]:
if year == 2023:
    brampton_hand_count_2023.columns = ['geocoded_city', 'geocoded_province', 'counterpoint_id',
       'counterpoint_name', 'time', 'date',
       'start_datetime',
       'end_datetime', 'easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']
    
    #assume empty cells are zero (trails)
    brampton_hand_count_2023= brampton_hand_count_2023.fillna(0)

    # concatenate brampton and hand count data
    paper_count = pd.concat([paper_count, brampton_hand_count_2023])
    
    original_shape += brampton_hand_count_2023.shape[0]

In [47]:
paper_count.shape

(32, 13)

In [48]:
# converts non-numbers to NA
paper_count['counterpoint_id'] = pd.to_numeric(paper_count['counterpoint_id'], errors='coerce')

In [49]:
# how many entries have a valid counterpoint id
paper_count[~paper_count['counterpoint_id'].isna()].shape

(29, 13)

In [50]:
paper_count['geocoded_city'] = paper_count['geocoded_city'].str.strip()

In [51]:
print("{} paper count sessions were submitted, of these {} could be assigned a counterpoint ID.".format(paper_count.shape[0],
                                                                               paper_count[~paper_count['counterpoint_id'].isna()].shape[0]))                         

32 paper count sessions were submitted, of these 29 could be assigned a counterpoint ID.


In [52]:
# assign a survey session id to hand count data
paper_count['survey_session_id'] = np.arange(1, paper_count.shape[0] + 1)

In [53]:
paper_count[['easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']] = paper_count[['easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']].fillna(0)

In [54]:
# in 2024 the start_datetime and end_datetime columns are just the time
if year == 2024:
    paper_count['start_datetime'] = pd.to_datetime(paper_count['date'] + ' ' + paper_count['start_datetime'])
    paper_count['end_datetime'] = pd.to_datetime(paper_count['date'] + ' ' + paper_count['end_datetime'])

In [55]:
# remove counts present in the hand count data that are also in the app data
paper_count['counterpoint_id'] = pd.to_numeric(paper_count['counterpoint_id'])

In [56]:
app_data_agg = app_data_filtered.groupby(['counterpoint_id', 'counterpoint_name', 'survey_session_id', 'lat', 
                         'lng', 'geocoded_city', 'start_datetime', 'start_datetime_hourofday', 
                                          'elapsed_time_seconds'])[['easy_bike_total', 'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']].sum().reset_index()

In [57]:
# shared counterpoint ids between hand and app data
possible_duplicates = paper_count[['survey_session_id', 'geocoded_city', 'counterpoint_id',
       'location_notes', 'easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total', 'date',
       'start_datetime', 'end_datetime']].merge(app_data_agg[['survey_session_id', 'counterpoint_id', 'counterpoint_name', 'start_datetime',
                                                                  'easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']].drop_duplicates(), on = 'counterpoint_id', suffixes = ['_hand', '_app'])

In [58]:
possible_duplicates['hand_datetime'] = pd.to_datetime(possible_duplicates['start_datetime_hand'])
possible_duplicates['app_datetime'] = pd.to_datetime(possible_duplicates['start_datetime_app'])

In [59]:
# narrow it down to shared day
possible_duplicates = possible_duplicates[possible_duplicates['hand_datetime'].dt.day == possible_duplicates['app_datetime'].dt.day]

In [60]:
# narrow it down to starting time within 1-2 hours of each other
if year == 2023:
    cutoff_time = 3600 # seconds
if year == 2024:
    cutoff_time = 5400 # seconds - one count is a definite duplicate because of app crash
possible_duplicates = possible_duplicates[np.abs((possible_duplicates['hand_datetime']
                                                  - possible_duplicates['app_datetime']).dt.total_seconds()) < cutoff_time]

In [61]:
# the red deer one (2023) looks like two people counted at the exact same time - in different directions or duplicate count? 
# first and last (2023) are definitely duplicates - keep app data
possible_duplicates[['geocoded_city', 'counterpoint_id', 'easy_bike_total_hand', 'easy_pedestrian_total_hand',
       'easy_personal_vehicle_total_hand', 'easy_oversized_total_hand', 'easy_bike_total_app',
       'easy_pedestrian_total_app', 'easy_personal_vehicle_total_app',
       'easy_oversized_total_app', 'hand_datetime', 'app_datetime']]

,geocoded_city,counterpoint_id,easy_bike_total_hand,easy_pedestrian_total_hand,easy_personal_vehicle_total_hand,easy_oversized_total_hand,easy_bike_total_app,easy_pedestrian_total_app,easy_personal_vehicle_total_app,easy_oversized_total_app,hand_datetime,app_datetime
8,Vancouver,5304.0,133.0,45.0,1.0,0.0,133,45,1,0,2024-06-06 14:00:00,2024-06-06 14:00:16
13,Vancouver,8231.0,134.0,32.0,0.0,0.0,134,32,0,0,2024-06-06 14:03:00,2024-06-06 14:03:08
20,Waterloo,5171.0,83.0,11.0,0.0,0.0,83,11,0,0,2024-06-04 07:07:00,2024-06-04 07:07:41
21,Edmonton,6516.0,24.0,33.0,2.0,0.0,14,15,1,0,2024-06-04 07:00:00,2024-06-04 07:52:29
26,Ottawa,6445.0,41.0,2.0,3.0,0.0,41,2,3,0,2024-06-04 13:57:00,2024-06-04 15:24:13


In [62]:
# keep the larger of app or hand count data
columns = ['easy_bike_total',
       'easy_pedestrian_total', 'easy_personal_vehicle_total',
       'easy_oversized_total']
suffixes = ['_hand', '_app']

for col in columns:
    possible_duplicates[col + '_diff'] = possible_duplicates[col + suffixes[0]].astype('int') - possible_duplicates[col + suffixes[1]].astype('int')
    
for i, val in enumerate(np.sum(possible_duplicates[['easy_bike_total_diff',
       'easy_pedestrian_total_diff', 'easy_personal_vehicle_total_diff',
       'easy_oversized_total_diff']], axis = 1)):
    
    if val <= 0 : # app count is the same or greater than paper count, keep the app count and drop the corresponding paper count
        survey_session_id = possible_duplicates.iloc[i]['survey_session_id_hand']
        paper_count = paper_count[paper_count['survey_session_id'] != survey_session_id] 
        rows_removed += 1
    else: # keep the hand count and drop the app count
        survey_session_id = possible_duplicates.iloc[i]['survey_session_id_app']
        app_data_filtered = app_data_filtered[app_data_filtered['survey_session_id'] != survey_session_id] 

In [63]:
possible_duplicates.shape

(5, 24)

In [64]:
print("{} paper counts were identified that matched a count location and time in the app and paper data.".format(possible_duplicates.shape[0]))

5 paper counts were identified that matched a count location and time in the app and paper data.


 We are aware of cases where volunteers counted with the app and also submitted their count by email due to app issues. In these cases, the count that had a higher total count was retained. (starting within two hours of each other) 

Change province abbreviations to be consistent with app data

In [65]:
paper_count['geocoded_province'].str.strip().sort_values().unique()

array(['Aberta', 'Alberta', 'Manitoba', 'Manitobia', 'New Brunswick',
       'Ontario', 'PEI', 'Quebec', 'Saskachewan'], dtype=object)

In [66]:
app_data_filtered['geocoded_province'].sort_values().unique()

array(['Alberta', 'British Columbia', 'Manitoba',
       'New Brunswick / Nouveau-Brunswick', 'Newfoundland and Labrador',
       'Northwest Territories', 'Nova Scotia', 'Ontario',
       'Prince Edward Island', 'Québec', 'Saskatchewan', 'Yukon'],
      dtype=object)

In [67]:
# 2023
if year == 2023:
    province_colnames = ['AB', 'BC', 'MB', 'NB', 'NS', 'ON', 'PE', 'SK', 'YT']
    province_names = ['Alberta', 'British Columbia', 'Manitoba', 'New Brunswick',
            'Nova Scotia', 'Ontario',
           'Prince Edward Island', 'Saskatchewan', 'Yukon']

# 2024 - fix typos in province names
if year == 2024:
    province_colnames = ['Aberta', 'Alberta', 'Manitoba', 'Manitobia', 'New Brunswick',
       'Ontario', 'PEI', 'Quebec', 'Saskachewan']
    province_names = ['Alberta','Alberta', 'Manitoba', 'Manitoba', 'New Brunswick',
                  'Ontario', 'Prince Edward Island', 'Quebec',    'Saskatchewan']

In [68]:
province_dict = dict(zip(province_colnames, province_names))

In [69]:
paper_count['geocoded_province'] = paper_count['geocoded_province'].str.strip().map(province_dict)


In [70]:
assert not np.all(paper_count['geocoded_province'].isna())

In [71]:
paper_count['start_datetime_dayofmonth'] = pd.to_datetime(paper_count['start_datetime']).dt.day

In [72]:
paper_count['easy_bike_total'] = paper_count['easy_bike_total'].astype('int')
#paper_count['easy_pedestrian_total'] = paper_count['easy_pedestrian_total'].astype('int')
#paper_count['easy_personal_vehicle_total'] = paper_count['easy_personal_vehicle_total'].astype('int')
#paper_count['easy_oversized_total'] = paper_count['easy_oversized_total'].astype('int')

In [73]:
assert paper_count.shape[0] == original_shape - rows_removed

In [74]:
rows_removed

4

In [75]:
paper_count[~paper_count['counterpoint_id'].isna()].shape

(25, 15)

## Save data

In [76]:
all_data = pd.concat([app_data_filtered, paper_count])
all_data.to_csv("data/%s_cleaned_data.csv" %year, index = None)
short_sessions.to_csv("data/%s_short_counts.csv" %year, index = None)